# Lab 2 · Inference

After Christmas, Isar Coffee gets hundreds of return and warranty claims – through the returns form
of the online shop (shop rule 4 in Lab 1), and by email from customers who bought at the market stand. For each claim
an AI step makes the decision (refund, replacement, reject or ask for more info) and stores it
in the returns database. The care team then sends one of its standard replies for that decision.

The AI step works and its decisions are correct. But it takes more than half a minute per claim –
while a customer may be waiting on the phone – and it is expensive. In this lab you
measure it, try the levers that make it faster and cheaper one at a time, and then combine them –
without the decisions getting worse.

| Part | What you do | Time |
|---|---|---|
| 0 | Connect | 3 min |
| 1 | Measure a single call | 5 min |
| 2 | Compare the three models | 5 min |
| 3 | Run the current version (v0) | 10 min |
| 4 | ✏️ Try the levers one by one | 25 min |
| 5 | ✏️ Combine them in a sprint | 15 min |
| ⭐ | Extensions | as homework |

Cells marked **▶ RUN** only need to be run. Cells marked **✏️ YOUR TURN** are where you work.
Cells that cost more than a cent say so. If you run every cell once, the lab costs about $0.50.

## Part 0 · Connect

One thing to know for this lab: **Haiku runs without thinking** – it answers straight away, which
makes it the fastest and cheapest option. Sonnet and Opus think before they answer, as usual.

In [ ]:
# ▶ RUN
import datetime
import json
import pathlib
import re
import statistics
import sys
import time
from concurrent.futures import ThreadPoolExecutor

_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "labs_common.py").exists())
sys.path.insert(0, str(_root))
from labs_common import connect

lab = connect()
client = lab.make_client(timeout=600, haiku_thinks=False)   # in this lab, Haiku answers without thinking
HERE = _root / "02_inference"

## Part 1 · Measure a single call

> **Goal:** know the three numbers you'll optimise. · **Time:** 5 min · **You edit:** nothing

- **TTFT** (time to first token): how long until the first word appears.
- **Total time**: how long until the answer is complete.
- **Cost**: what one call costs, including cached tokens (Part 4 explains those).

`timed_call` sends one request as a stream, notes when the first token arrives and when the
last one does, and works out the cost.

In [ ]:
# ▶ RUN – the measuring helper


def text_of(response):
    return "".join(b.text for b in response.content if b.type == "text")


def timed_call(**params):
    """One streamed API call. Returns (response, ttft_seconds, total_seconds, cost_usd)."""
    start = time.perf_counter()
    ttft = None
    with client.messages.stream(**params) as stream:
        for event in stream:
            if ttft is None and event.type == "content_block_delta":
                ttft = time.perf_counter() - start
        response = stream.get_final_message()
    total = time.perf_counter() - start
    return response, ttft or total, total, lab.cost(params["model"], response.usage)


r, ttft, total, cost = timed_call(model=lab.HAIKU, max_tokens=300,
                                  messages=[{"role": "user", "content": "In one sentence: what is a moka pot?"}])
print(text_of(r))
print(f"\nTTFT {ttft:.2f}s · total {total:.2f}s · {r.usage.output_tokens} output tokens · ${cost:.5f}")

## Part 2 · Compare the three models

> **Goal:** see how much speed and price differ between models. · **Time:** 5 min · **You edit:** nothing

Same question, three models, two runs each. This is often the biggest lever of all: before you
optimise *how* you call a model, decide *which* model a task needs.

In [ ]:
# ▶ RUN – about 30 seconds, costs about $0.02
QUESTION = "Explain in two sentences why coffee from a moka pot is not real espresso."
print(f"{'model':38} {'TTFT':>6} {'total':>7} {'tokens':>7} {'$ per 1,000 calls':>18}")
for model in (lab.HAIKU, lab.SONNET, lab.OPUS):
    for _ in range(2):
        r, ttft, total, cost = timed_call(model=model, max_tokens=1000,
                                          messages=[{"role": "user", "content": QUESTION}])
        print(f"{model:38} {ttft:5.2f}s {total:6.2f}s {r.usage.output_tokens:7} {cost * 1000:17.2f}")

The prices per million tokens (input / output) are $0.10 / $0.50 for Haiku, $2 / $10 for Sonnet
and $4 / $20 for Opus. Output tokens cost five times as much as input tokens – keep that in mind
for Part 4.

Why do Sonnet and Opus need so many output tokens for two sentences? They think before they answer.
You don't see the thinking here, but it is billed as output tokens and takes time. Haiku skips it in
this lab. Less output – shorter answers, less thinking – is one of the strongest levers you have.

## Part 3 · Run the current version (v0)

> **Goal:** get the baseline and spot what's wasteful. · **Time:** 10 min ·
> **You edit:** your notes · **Done when:** you have written down three things that look wasteful.

**The handbook.** The care team decides claims with a handbook: return rules, warranty,
defect-or-misuse signs for every product, prices and 36 past cases. It is the same for
every single claim. Open `handbook.md` if you're curious – you don't need to learn it.

**The claims.** Fourteen sample claims, each with the correct decision filled in by the care
team. That's our accuracy check, like the eval in Lab 1.

In [ ]:
# ▶ RUN – load the handbook and the claims
HANDBOOK = (HERE / "handbook.md").read_text(encoding="utf-8")
print(f"Handbook: {len(HANDBOOK):,} characters, about {len(HANDBOOK) // 4:,} tokens\n")

CLAIMS = [
    {"id": "R-01", "channel": "form", "customer": "Lena Maier", "received": "2027-01-08",
     "order": "ORD-10231: Decaf (250 g) €9.90 · order total €9.90, shipping €4.90 charged",
     "claimed": "Decaf (250 g)", "delivered": "2027-01-03", "opened": "no", "dropdown": "changed my mind", "system_refund": 14.80, "description": "",
     "gold": {"decision": "refund", "refund_eur": 14.80, "reason": "within_return_window"}},
    {"id": "R-02", "channel": "form", "customer": "Tobias Brandl", "received": "2027-01-04",
     "order": "ORD-10198: Espresso Classico 1 kg €29.90 · order total €29.90, shipping €4.90 charged",
     "claimed": "Espresso Classico 1 kg", "delivered": "2026-12-28", "opened": "yes", "dropdown": "changed my mind", "system_refund": 34.80,
     "description": "Bought the wrong coffee by mistake.",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "opened_coffee"}},
    {"id": "R-03", "channel": "email", "customer": "Sophie Kraus", "received": "2026-12-30",
     "email": "Hi, my order IC-2291 arrived yesterday. The bag of Isar Crema was torn at the side and the "
              "beans smell stale, not fresh at all. Can you do something? Sophie",
     "gold": {"decision": "replacement", "refund_eur": 0, "reason": "quality_issue"}},
    {"id": "R-04", "channel": "form", "customer": "Markus Weber", "received": "2027-01-06",
     "order": "ORD-09412: Moka Pot €34.90, Espresso Classico (250 g) €8.90 · order total €43.80, free shipping",
     "claimed": "Moka Pot", "delivered": "2026-10-05", "opened": "yes", "dropdown": "not working",
     "description": "The handle came off while I was pouring.",
     "gold": {"decision": "replacement", "refund_eur": 0, "reason": "warranty_defect"}},
    {"id": "R-05", "channel": "email", "customer": "Julia Schmid", "received": "2027-01-05",
     "email": "Hello, I ordered a moka pot from you in November (order IC-1874). I put it in the dishwasher "
              "a few times and now it's black inside and the coffee tastes metallic. I'd like a new one please. Julia",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "misuse"}},
    {"id": "R-06", "channel": "form", "customer": "Felix Huber", "received": "2027-01-04",
     "order": "ORD-09877: Hand Grinder €49.00 · order total €49.00, free shipping",
     "claimed": "Hand Grinder", "delivered": "2026-11-20", "opened": "no", "dropdown": "changed my mind", "system_refund": 49.00, "description": "",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "outside_return_window"}},
    {"id": "R-07", "channel": "form", "customer": "Anna Lehner", "received": "2027-01-06",
     "order": "ORD-10120: Moka Pot €34.90, Espresso Cups (set of 2) €19.90 · order total €54.80, free shipping",
     "claimed": "Espresso Cups (set of 2)", "delivered": "2026-12-27", "opened": "no", "dropdown": "changed my mind", "system_refund": 19.90,
     "description": "",
     "gold": {"decision": "refund", "refund_eur": 19.90, "reason": "within_return_window"}},
    {"id": "R-08", "channel": "form", "customer": "Stefan Bauer", "received": "2027-01-05",
     "order": "ORD-08502: Milk Frother €24.90 · order total €24.90, shipping €4.90 charged",
     "claimed": "Milk Frother", "delivered": "2026-09-01", "opened": "yes", "dropdown": "not working",
     "description": "Stopped working, also with new batteries.",
     "gold": {"decision": "replacement", "refund_eur": 0, "reason": "warranty_defect"}},
    {"id": "R-09", "channel": "email", "customer": "Maria Gruber", "received": "2027-01-07",
     "email": "Hi, I bought one of your moka pots at the Viktualienmarkt stand last summer. Now the handle is "
              "loose and wobbles when I pour. Can you help? I don't think I kept the receipt. Maria",
     "gold": {"decision": "need_info", "refund_eur": 0, "reason": "missing_proof_of_purchase"}},
    {"id": "R-10", "channel": "email", "customer": "Ben Schulz", "received": "2027-01-08",
     "email": "Hi, my order IC-2290 arrived on 9 December. I haven't opened the Hand Grinder and would like to "
              "return it – it was the only thing in the order. Ben",
     "gold": {"decision": "refund", "refund_eur": 49.00, "reason": "within_return_window"}},
    {"id": "R-11", "channel": "email", "customer": "Lukas Wolf", "received": "2027-01-07",
     "email": "Hi, I want to return my whole order IC-2350 – a bag of Isar Crema and a bag of Espresso Classico, "
              "both still sealed. It arrived on 30 December. Lukas",
     "gold": {"decision": "refund", "refund_eur": 23.30, "reason": "within_return_window"}},
    {"id": "R-12", "channel": "form", "customer": "Nina Roth", "received": "2027-01-07",
     "order": "ORD-10302: 2 × Decaf (250 g) €19.80 · order total €19.80, shipping €4.90 charged",
     "claimed": "2 × Decaf (250 g)", "delivered": "2027-01-02", "opened": "one of the two bags", "dropdown": "other", "system_refund": 24.70,
     "description": "I opened one bag before I noticed I had ordered decaf by mistake. The other one is still "
                    "sealed. I'm sending both back.",
     "gold": {"decision": "refund", "refund_eur": 9.90, "reason": "within_return_window"}},
    {"id": "R-13", "channel": "form", "customer": "Paul Wagner", "received": "2027-01-07",
     "order": "ORD-10144: 2 × Isar Crema (250 g) €19.00 · order total €19.00, shipping €4.90 charged",
     "claimed": "2 × Isar Crema (250 g)", "delivered": "2026-12-26", "opened": "no",
     "dropdown": "changed my mind", "system_refund": 23.90, "description": "",
     "gold": {"decision": "refund", "refund_eur": 23.90, "reason": "within_return_window"}},
    {"id": "R-14", "channel": "email", "customer": "Ole Brandt", "received": "2027-01-05",
     "email": "Hello, my Milk Frother (order IC-2009) stopped working, even with new batteries. I always clean "
              "it really well – after every use I rinse the whole frother under the tap. It's only five months old. Ole",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "misuse"}},
]


def claim_text(claim):
    """The claim as the model sees it."""
    if claim["channel"] == "email":
        return (f"Claim {claim['id']} · email received {claim['received']} from {claim['customer']}\n\n"
                f"{claim['email']}")
    days = (datetime.date.fromisoformat(claim["received"]) - datetime.date.fromisoformat(claim["delivered"])).days
    return (f"Claim {claim['id']} · web form received {claim['received']}\n"
            f"Order: {claim['order']}\nItem claimed: {claim['claimed']}\n"
            f"Delivered: {claim['delivered']} ({days} days ago)\n"
            f"Opened or used: {claim['opened']}\nReason (dropdown): {claim['dropdown']}\n"
            f"Description: {claim['description'] or '–'}"
            + (f"\nRefund if the whole return is accepted (worked out by the shop system): €{claim['system_refund']:.2f}"
               if "system_refund" in claim else ""))


print(claim_text(CLAIMS[3]))

**The check.** Every claim has three fields to get right: `decision`, `refund_eur` and `reason`.
Like a task in Lab 1, a claim only counts as correct if all three are right – a refund with the wrong
amount is a wrong decision. **Accuracy must stay at 90 % or more** – a faster pipeline that refunds
dishwasher-damaged pots is not an improvement.

In [ ]:
# ▶ RUN – the grader and the helpers that run a pipeline over all claims


def grade(record, gold):
    record = record or {}
    try:
        refund_ok = abs(float(record.get("refund_eur", -1)) - gold["refund_eur"]) < 0.01
    except (TypeError, ValueError):
        refund_ok = False
    return {
        "decision": str(record.get("decision", "")).strip().lower() == gold["decision"],
        "refund_eur": refund_ok,
        "reason": str(record.get("reason", "")).strip().lower() == gold["reason"],
    }


def run_claims(pipeline, claims=CLAIMS, workers=0, warm_first=True):
    """Run a pipeline over the claims. workers=0 means one after another."""

    def one(claim):
        out = pipeline(claim)
        checks = grade(out["record"], claim["gold"])
        return {"id": claim["id"], "record": out["record"], "checks": checks,
                "seconds": out["seconds"], "calls": len(out["calls"]),
                "cost": sum(lab.cost(m, u) for m, u in out["calls"]),
                "output_tokens": sum(u.output_tokens for _, u in out["calls"]),
                "cache_read": sum(getattr(u, "cache_read_input_tokens", 0) or 0 for _, u in out["calls"]),
                "cache_write": sum(getattr(u, "cache_creation_input_tokens", 0) or 0 for _, u in out["calls"])}

    start = time.perf_counter()
    if workers:
        rows = [one(claims[0])] if warm_first else []        # one claim first fills the cache
        rest = claims[1:] if warm_first else claims
        with ThreadPoolExecutor(max_workers=workers) as pool:
            rows += list(pool.map(one, rest))
    else:
        rows = [one(c) for c in claims]
    return summarise(rows, time.perf_counter() - start)


def summarise(rows, wall):
    correct = sum(all(r["checks"].values()) for r in rows)        # a claim counts only if all 3 fields are right
    return {"rows": rows, "accuracy": correct / len(rows),
            "p50": statistics.median(r["seconds"] for r in rows),
            "cost_per_claim": sum(r["cost"] for r in rows) / len(rows), "wall": wall}


def report(rep, label):
    print(f"── {label}")
    print(f"{'claim':6} {'decision':12} {'€':>6} {'reason':26} {'ok':4} {'time':>6} {'calls':>5} {'cost':>8}")
    for r in rep["rows"]:
        rec = r["record"] or {}
        ok = sum(r["checks"].values())
        print(f"{r['id']:6} {str(rec.get('decision', '?')):12} {str(rec.get('refund_eur', '?')):>6} "
              f"{str(rec.get('reason', '?')):26} {ok}/3  {r['seconds']:5.1f}s {r['calls']:5} ${r['cost']:.4f}")
    print(f"\naccuracy {rep['accuracy']:.0%} · p50 {rep['p50']:.1f}s per claim · "
          f"${rep['cost_per_claim']:.4f} per claim · {rep['wall']:.0f}s for all claims")


print("Grader ready.")

**v0 – the version you inherit.** Read it the way you'd read code someone else wrote: not to
judge it, but to find where the time and money go.

In [ ]:
# ▶ RUN – inference_v0, exactly as it was built


def v0_system(claim):
    # The letter should greet the customer by name, so the name goes in the system prompt.
    return f"Today is {claim['received']}. You are writing to {claim['customer']}.\n\n" + HANDBOOK


CHECK_WINDOW = "Check only this: is the claim within the 30-day return window? Count the days. Explain your reasoning."
CHECK_OPENED = "Check only this: was the item opened or used, and does the exception for quality problems apply? Explain."
CHECK_WARRANTY = "Check only this: is this a warranty case, and is it a defect or misuse? Explain."

V0_FINAL = """Your checks:

{notes}

Now write the reply letter to the customer. At the very end, add these three lines:
DECISION: refund, replacement, reject or need_info
REFUND: the amount in euros, e.g. 14.80 (0 if nothing is refunded)
REASON: one reason code from the handbook"""


def parse_letter(text):
    """Pull DECISION / REFUND / REASON out of the letter (the last time each label appears)."""
    def find(label, pattern):
        found = re.findall(rf"{label}\s*\**\s*:[\s*`'\"]*€?\s*({pattern})", text, re.IGNORECASE)
        return found[-1] if found else None
    refund = find("REFUND", r"\d+(?:[.,]\d+)?")
    return {"decision": (find("DECISION", r"[a-z_]+") or "").lower(),
            "refund_eur": float(refund.replace(",", ".")) if refund else None,
            "reason": (find("REASON", r"[a-z_]+") or "").lower()}


def inference_v0(claim, model=None):
    model = model or lab.OPUS                     # v0 uses Opus for every claim
    system = v0_system(claim)
    claim_info = claim_text(claim)
    start = time.perf_counter()

    # Call 1 of 4 – is the claim within the return window?
    r1 = client.messages.create(model=model, max_tokens=4000, system=system,
                                messages=[{"role": "user", "content": claim_info + "\n\n" + CHECK_WINDOW}])

    # Call 2 of 4 – was the item opened or used?
    r2 = client.messages.create(model=model, max_tokens=4000, system=system,
                                messages=[{"role": "user", "content": claim_info + "\n\n" + CHECK_OPENED}])

    # Call 3 of 4 – is it a defect or misuse?
    r3 = client.messages.create(model=model, max_tokens=4000, system=system,
                                messages=[{"role": "user", "content": claim_info + "\n\n" + CHECK_WARRANTY}])

    # Call 4 of 4 – write the letter, using the three answers above
    notes = text_of(r1) + "\n\n" + text_of(r2) + "\n\n" + text_of(r3)
    r4 = client.messages.create(model=model, max_tokens=4000, system=system,
                                messages=[{"role": "user", "content": claim_info + "\n\n" + V0_FINAL.format(notes=notes)}])

    letter = text_of(r4)
    return {"record": parse_letter(letter), "letter": letter,
            "calls": [(model, r.usage) for r in (r1, r2, r3, r4)],
            "seconds": time.perf_counter() - start}


print("v0 ready.")

Running v0 on all fourteen claims takes about 9 minutes and costs about $2. You don't need to do
that: we ran it for you and saved the result. Run it on **one** claim to see it for yourself.

In [ ]:
# ▶ RUN – v0 on one claim (about 1 minute, about $0.15)
one = inference_v0(CLAIMS[3])
cost = sum(lab.cost(m, u) for m, u in one["calls"])
print(one["letter"][-1500:])
print(f"\n→ record: {one['record']}")
print(f"→ {len(one['calls'])} calls · {one['seconds']:.0f}s · ${cost:.3f} · "
      f"{sum(u.output_tokens for _, u in one['calls']):,} output tokens")

In [ ]:
# ▶ RUN – the recorded v0 run over all fourteen claims
RUN_V0_LIVE = False   # set to True to run it yourself (about 9 minutes, about $2)

if RUN_V0_LIVE:
    BASELINE = run_claims(inference_v0)
else:
    saved = json.loads((HERE / "v0_baseline.json").read_text())
    BASELINE = summarise(saved["rows"], saved["wall"])
report(BASELINE, "v0 – the baseline")

**Before you read on:** write down three things in v0 that look wasteful – slow, expensive or
simply not needed. Look at the code and at the numbers above.

In [ ]:
# ✏️ YOUR TURN – your notes (just text, nothing to run)
my_notes = """
1.
2.
3.
"""

<details><summary>What we found (open after writing your notes)</summary>

Each finding is one lever in Part 4:

1. **Four calls per claim** – three separate checks and a fourth call for the letter. Each call
   sends the whole handbook again and waits for a full answer. → Lever 1
2. **It writes a letter nobody asked for.** The database only needs four fields. Output tokens
   are the most expensive part, and the regex that digs the decision out of the letter can break. → Lever 2
3. **The handbook is never cached.** It is identical for every claim, but the date and the
   customer's name sit *in front of* it, so every request looks new. → Lever 3
4. **Opus for every claim**, even "unopened, changed my mind, 5 days ago". → Lever 4
5. **Nothing appears until everything is finished.** → Lever 5
6. **Claims run one after another.** → Lever 6
7. **Everything is billed at full price**, even claims that could wait until tomorrow. → Lever 7

</details>

## Part 4 · The levers

> **Goal:** see what each lever does on its own. · **Time:** 25 min ·
> **You edit:** three small ✏️ cells · **Done when:** you've run every lever and its self-check shows ✓.

To keep costs low, the levers are shown on Sonnet and on a single claim where possible.

### Lever 1 · Fewer round trips

Every call pays the full price: it sends the handbook, waits, and generates an answer. v0 makes
four of them per claim. The checks don't need separate calls – one call can apply all the rules.

In [ ]:
# ▶ RUN – v0's four calls against one call, both on Sonnet (about 40 s, about $0.08)

ONE_CALL = """Decide this claim using the handbook. Then write the reply letter to the customer.
At the very end, add these three lines:
DECISION: refund, replacement, reject or need_info
REFUND: the amount in euros, e.g. 14.80 (0 if nothing is refunded)
REASON: one reason code from the handbook"""


def one_call(claim, model):
    start = time.perf_counter()
    r = client.messages.create(model=model, max_tokens=4000, system=v0_system(claim),
                               messages=[{"role": "user", "content": claim_text(claim) + "\n\n" + ONE_CALL}])
    return {"record": parse_letter(text_of(r)), "calls": [(model, r.usage)], "seconds": time.perf_counter() - start}


for name, out in (("4 calls (v0)", inference_v0(CLAIMS[3], model=lab.SONNET)),
                  ("1 call", one_call(CLAIMS[3], lab.SONNET))):
    print(f"{name:13} {out['seconds']:5.1f}s · ${sum(lab.cost(m, u) for m, u in out['calls']):.4f} · {out['record']}")

### Lever 2 · Structured output

The database needs four fields, not a letter. Without a fixed format the answer can be as long
as the model likes – v0 writes a whole letter, and the decision has to be dug out with a regex.
With `output_config` you give the API a JSON schema, and the answer comes back as exactly that
JSON: short, always the same shape, ready to store. The schema below is missing one field.

In [ ]:
# ✏️ YOUR TURN – add the missing "reason" field to the schema

REASONS = ["within_return_window", "outside_return_window", "opened_coffee", "quality_issue",
           "warranty_defect", "misuse", "missing_proof_of_purchase"]

RECORD_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "claim_id": {"type": "string"},
            "decision": {"type": "string", "enum": ["refund", "replacement", "reject", "need_info"]},
            "refund_eur": {"type": "number"},
        },
        "required": ["claim_id", "decision", "refund_eur"],
        "additionalProperties": False,
    },
}

<details><summary>💡 Hint</summary>

Look at how `decision` is defined. `reason` works the same way, with the list `REASONS`.
Don't forget the `required` list.

</details>

In [ ]:
# ▶ RUN – self-check, then one claim with the schema (about 10 s)
DECIDE = "Decide this claim using the handbook. Return only the database record."
_props = RECORD_SCHEMA["schema"]["properties"]
if "reason" not in _props:
    print("✗ The schema has no 'reason' field yet.")
elif sorted(_props["reason"].get("enum", [])) != sorted(REASONS):
    print("✗ 'reason' should only allow the codes in REASONS.")
elif "reason" not in RECORD_SCHEMA["schema"]["required"]:
    print("✗ 'reason' is not in the required list, so the model may leave it out.")
else:
    print("✓ Schema complete.\n")
    r, ttft, total, cost = timed_call(model=lab.SONNET, max_tokens=2000, system=v0_system(CLAIMS[3]),
                                      messages=[{"role": "user", "content": claim_text(CLAIMS[3]) + "\n\n" + DECIDE}],
                                      output_config={"format": RECORD_SCHEMA})
    print(text_of(r))
    print(f"\n{r.usage.output_tokens} output tokens · {total:.1f}s · ${cost:.4f}")

### Lever 3 · Prompt caching

The handbook is the same for every claim. With **prompt caching** the API keeps the processed
handbook for 5 minutes. The first request writes it to the cache (a bit more expensive), every
request after that reads it back at a fraction of the price.

The catch: the cache only matches if **the beginning of the request is byte-for-byte the same**.
v0 puts the date and the customer's name *in front of* the handbook, so every request starts
differently and nothing is ever reused.

Fix `build_request` below: the handbook goes first, marked with `cache_control`; everything that
changes per claim goes after it, in `claim_part`. With the JSON record, the name isn't needed at all.

In [ ]:
# ✏️ YOUR TURN – put the handbook first and cache it


def build_request(claim):
    """Returns (system, claim_part). The instruction is added after claim_part by whoever sends the request."""
    system = f"Today is {claim['received']}. You are writing to {claim['customer']}.\n\n" + HANDBOOK
    claim_part = claim_text(claim)
    return system, claim_part

<details><summary>💡 Hint</summary>

`system` can be a list of blocks instead of a string:
`[{"type": "text", "text": HANDBOOK, "cache_control": {"type": "ephemeral"}}]`.
The date can go at the start of `claim_part`, e.g. `f"Today is {claim['received']}.\n\n" + claim_text(claim)`.

</details>

In [ ]:
# ▶ RUN – self-check, then the same request twice (about 20 s)
_system, _part = build_request(CLAIMS[3])
_user = _part + "\n\n" + DECIDE
if isinstance(_system, str):
    print("✗ system is still a plain string. Make it a list with one block for the handbook.")
elif _system[0].get("text") != HANDBOOK or "cache_control" not in _system[0]:
    print("✗ The first system block should be exactly the handbook, with cache_control.")
elif CLAIMS[3]["received"] in json.dumps(_system):
    print("✗ The date is still in the system prompt. Move it to the user message.")
else:
    print("✓ Handbook first and cached.\n")
    for attempt in ("first request", "second request"):
        r, ttft, total, cost = timed_call(model=lab.SONNET, max_tokens=2000, system=_system,
                                          messages=[{"role": "user", "content": _user}],
                                          output_config={"format": RECORD_SCHEMA})
        u = r.usage
        print(f"{attempt:15} cache write {u.cache_creation_input_tokens or 0:5} · cache read "
              f"{u.cache_read_input_tokens or 0:5} · TTFT {ttft:.2f}s · ${cost:.4f}")

On the second request, `cache read` should be about the size of the handbook. If both show 0,
something before the handbook still changes between requests.

### Lever 4 · Routing

Haiku is many times cheaper than Sonnet. Is it good enough? It depends on the claim:

- **Web-form claims** arrive with clean data – the shop system works out how many days ago the
  order was delivered and what the refund would be. "Unopened, changed my mind, 5 days ago" is easy.
- **Emails and special cases** have to be worked out: count the days from "arrived on 9 December",
  add up items and shipping, decide which part of a return is valid.

So send the easy claims to Haiku and the rest to Sonnet. But which ones are easy? Measure first.

In [ ]:
# ▶ RUN – every claim on Haiku, then on Sonnet (about 1 min, about $0.03)


def decide_with(model):
    """A pipeline that decides every claim with one model."""
    def pipeline(claim):
        system, part = build_request(claim)
        start = time.perf_counter()
        r = client.messages.create(model=model, max_tokens=2000, system=system,
                                   messages=[{"role": "user", "content": part + "\n\n" + DECIDE}],
                                   output_config={"format": RECORD_SCHEMA})
        return {"record": json.loads(text_of(r)), "calls": [(model, r.usage)], "seconds": time.perf_counter() - start}
    return pipeline


LEVER4_READY = "reason" in RECORD_SCHEMA["schema"]["properties"] and not isinstance(build_request(CLAIMS[0])[0], str)
if not LEVER4_READY:
    print("⚠️ Finish Lever 2 (schema) and Lever 3 (build_request) first – this lever uses both.")
for name, model in (("Haiku", lab.HAIKU), ("Sonnet", lab.SONNET)) if LEVER4_READY else ():
    rep = run_claims(decide_with(model), workers=6)
    wrong = [r["id"] for r in rep["rows"] if not all(r["checks"].values())]
    print(f"everything on {name:7} accuracy {rep['accuracy']:4.0%} · ${rep['cost_per_claim']:.5f} per claim · "
          f"p50 {rep['p50']:.1f}s · wrong: {', '.join(wrong) or '–'}")

Haiku costs a fraction – but it gets some claims wrong. Look them up with `print(claim_text(...))`:
what do the claims it gets wrong have in common, and what do the ones it gets right have in common?
Write a rule that only sends the safe ones to Haiku.

In [ ]:
# ✏️ YOUR TURN – which model gets a claim?


def pick_model(claim):
    return lab.SONNET

<details><summary>💡 Hint 1</summary>

Every claim has `claim["channel"]` ("form" or "email"). Web-form claims also have
`claim["dropdown"]`. Which of these does Haiku handle?

</details>

<details><summary>💡 Hint 2</summary>

Haiku is safe for web forms with "changed my mind": the shop system has already worked out the days
and the amount. Emails and "other" cases have to be worked out from the text.

</details>

In [ ]:
# ▶ RUN – self-check, then routed (about 30 s, about $0.02)
_small = {c["id"] for c in CLAIMS if pick_model(c) == lab.HAIKU}
if not LEVER4_READY:
    print("⚠️ Finish Lever 2 and Lever 3 first.")
elif _small != {"R-01", "R-02", "R-06", "R-07", "R-13"}:
    print(f"✗ Haiku gets {sorted(_small) or 'nothing'}. Compare with the claims Haiku got wrong above.")
else:
    print("✓ R-01, R-02, R-06, R-07 and R-13 go to Haiku, everything else to Sonnet.\n")


def routed(claim):
    return decide_with(pick_model(claim))(claim)


if LEVER4_READY:
    rep = run_claims(routed, workers=6)
    wrong = [r["id"] for r in rep["rows"] if not all(r["checks"].values())]
    print(f"routed              accuracy {rep['accuracy']:4.0%} · ${rep['cost_per_claim']:.5f} per claim · "
          f"p50 {rep['p50']:.1f}s · wrong: {', '.join(wrong) or '–'}")

Routing keeps Sonnet's accuracy and pays Sonnet's price only where it's needed.

### Lever 5 · Streaming

Streaming doesn't make a call finish sooner. It shows the first part earlier, while the rest is
still being written. That matters when someone is watching the screen. The cell below streams
v0's long letter and then the short JSON record.

In [ ]:
# ▶ RUN – a long and a short answer, streamed (about 30 s, about $0.02)
r, ttft, total, cost = timed_call(model=lab.SONNET, max_tokens=4000, system=v0_system(CLAIMS[4]),
                                  messages=[{"role": "user", "content": claim_text(CLAIMS[4]) + "\n\n" + ONE_CALL}])
print(f"letter:      first token after {ttft:5.2f}s · complete after {total:5.2f}s · {r.usage.output_tokens} output tokens")
_system, _part = build_request(CLAIMS[4])
r, ttft, total, cost = timed_call(model=lab.SONNET, max_tokens=2000, system=_system,
                                  messages=[{"role": "user", "content": _part + "\n\n" + DECIDE}],
                                  output_config={"format": RECORD_SCHEMA})
print(f"JSON record: first token after {ttft:5.2f}s · complete after {total:5.2f}s · {r.usage.output_tokens} output tokens")

With the letter, a reader sees text long before it's finished. With the short record, first and
last token arrive almost together – the model thinks first, then writes a few tokens. So streaming
matters for long answers that a person reads, and hardly at all for short records in a database.

### Lever 6 · Parallel, with a warm cache

Claims don't depend on each other, so they can run at the same time. One detail: the cache is
only filled once the first request has started answering. If all requests start at the same
moment, each one pays to write the cache. So: send one claim first, then the rest.

Each run below uses a fresh copy of the handbook, so every run starts with an empty cache.

In [ ]:
# ▶ RUN – all fourteen claims three ways (about 1–2 min, about $0.15)


def decide_fresh(tag):
    """Sonnet with a cached handbook copy that no earlier request has used."""
    system = [{"type": "text", "text": f"Handbook copy {tag}\n\n" + HANDBOOK, "cache_control": {"type": "ephemeral"}}]

    def pipeline(claim):
        start = time.perf_counter()
        r = client.messages.create(model=lab.SONNET, max_tokens=2000, system=system,
                                   messages=[{"role": "user", "content": f"Today is {claim['received']}.\n\n"
                                              + claim_text(claim) + "\n\n" + DECIDE}],
                                   output_config={"format": RECORD_SCHEMA})
        return {"record": json.loads(text_of(r)), "calls": [(lab.SONNET, r.usage)],
                "seconds": time.perf_counter() - start}
    return pipeline


for label, kwargs in (("one after another", {"workers": 0}),
                      ("parallel, cold cache", {"workers": 8, "warm_first": False}),
                      ("parallel, warm first", {"workers": 8, "warm_first": True})):
    rep = run_claims(decide_fresh(f"{label} {time.time()}"), **kwargs)
    writes = sum(r["cache_write"] for r in rep["rows"])
    print(f"{label:22} {rep['wall']:5.1f}s for all claims · cache writes {writes:6,} tokens · "
          f"accuracy {rep['accuracy']:.0%}")

"One after another" is cheap but slow. "Parallel, cold" is fast but writes the cache for many
claims. "Parallel, warm first" is a little slower than cold, because the first claim runs on its
own – but it writes the cache only once. With hundreds of claims, that's the one you want.

### Lever 7 · Batch

Claims that come in overnight don't need an answer in seconds. The **Batch API** takes up to
100,000 requests at once, processes them within 24 hours (usually much faster) and charges
**50 % less**. Same model, same prompt, same schema.

In [ ]:
# ▶ RUN – build the batch requests

batch_requests = []
for claim in CLAIMS:
    system, part = build_request(claim)
    batch_requests.append({"custom_id": claim["id"], "params": {
        "model": pick_model(claim), "max_tokens": 2000, "system": system,
        "messages": [{"role": "user", "content": part + "\n\n" + DECIDE}],
        "output_config": {"format": RECORD_SCHEMA}}})
print(f"{len(batch_requests)} requests ready.")

RUN_BATCH = False     # set to True to send them (needs your own Anthropic API key)
if RUN_BATCH and lab.via_gateway:
    print("Your company gateway doesn't offer the Batch API. The cost effect is the same: 50 % off.")
elif RUN_BATCH:
    batch = client.messages.batches.create(requests=batch_requests)
    print(f"Sent batch {batch.id}. Check later with client.messages.batches.retrieve('{batch.id}').")

## Part 5 · The sprint

> **Goal:** combine the levers into the fastest and cheapest pipeline that stays correct. ·
> **Time:** 15 min · **You edit:** `CONFIG` · **Done when:** you're happy with your numbers and
> have run the holdout once.

`inference_v1` starts with the same settings as v0. Switch levers on in `CONFIG` and run the
scorecard. It compares your run with the recorded v0 in two numbers: **how many times cheaper**
and **how many times faster** per claim. Both only count if accuracy stays at 90 % or more.
The cell after the scorecard draws all your runs side by side, so you can see what each change did.

One new setting: `effort` controls how much the model thinks before it answers. `"low"` means
less thinking – fewer output tokens and faster answers. Check that accuracy holds when you use it.

In [ ]:
# ✏️ YOUR TURN – your pipeline settings

CONFIG = {
    "one_call": False,      # Lever 1: one call instead of four
    "structured": False,    # Lever 2: JSON record instead of a letter
    "cache": False,         # Lever 3: build_request with the cached handbook
    "routing": False,       # Lever 4: pick_model instead of one model for everything
    "model": "opus",        # the model when routing is off: "haiku", "sonnet" or "opus"
    "effort": None,         # None, "low", "medium" or "high"
    "workers": 0,           # Lever 6: claims at the same time (0 = one after another)
}

In [ ]:
# ▶ RUN – inference_v1 and the scorecard. Read it from top to bottom: each step is one lever.
V0_CONFIG = {"one_call": False, "structured": False, "cache": False, "routing": False,
             "model": "opus", "effort": None, "workers": 0}
MODELS = {"haiku": lab.HAIKU, "sonnet": lab.SONNET, "opus": lab.OPUS}


def ask(model, system, text, **output_config):
    """One API call. output_config takes format= and effort=; empty values are left out."""
    output_config = {k: v for k, v in output_config.items() if v}
    extra = {"output_config": output_config} if output_config else {}
    return client.messages.create(model=model, max_tokens=4000, system=system,
                                  messages=[{"role": "user", "content": text}], **extra)


def inference_v1(claim):
    start = time.perf_counter()
    calls = []

    # Lever 4: which model?
    model = pick_model(claim) if CONFIG["routing"] else MODELS[CONFIG["model"]]

    # Lever 3: cached handbook first, or v0's request with the date and name in front?
    if CONFIG["cache"]:
        system, claim_part = build_request(claim)
    else:
        system, claim_part = v0_system(claim), claim_text(claim)

    # Lever 1: three separate checks first (like v0), or straight to the decision?
    notes = ""
    if not CONFIG["one_call"]:
        answers = []
        for check in (CHECK_WINDOW, CHECK_OPENED, CHECK_WARRANTY):   # v0's three separate checks
            r = ask(model, system, claim_part + "\n\n" + check, effort=CONFIG["effort"])
            calls.append((model, r.usage))
            answers.append(text_of(r))
        notes = "Your checks:\n\n" + "\n\n".join(answers) + "\n\n"

    # Lever 2: a JSON record, or a letter with the decision at the end?
    if CONFIG["structured"]:
        r = ask(model, system, claim_part + "\n\n" + notes + DECIDE,
                format=RECORD_SCHEMA, effort=CONFIG["effort"])
        record = json.loads(text_of(r))
    else:
        r = ask(model, system, claim_part + "\n\n" + notes + ONE_CALL, effort=CONFIG["effort"])
        record = parse_letter(text_of(r))
    calls.append((model, r.usage))

    return {"record": record, "calls": calls, "seconds": time.perf_counter() - start}


RUN_HISTORY = []          # every scorecard run on the sample claims, for the chart below


def config_label():
    """A short name for the current CONFIG, e.g. '1 call + JSON + cache · routed'."""
    parts = [name for key, name in (("one_call", "1 call"), ("structured", "JSON"), ("cache", "cache")) if CONFIG[key]]
    parts.append("routed" if CONFIG["routing"] else CONFIG["model"])
    if CONFIG["effort"]:
        parts.append(f"effort {CONFIG['effort']}")
    if CONFIG["workers"]:
        parts.append(f"{CONFIG['workers']} parallel")
    return " + ".join(parts)


def plot_runs(runs, title):
    """Bars per run, as multiples of v0: cheaper per claim, faster per claim, faster for all claims."""
    import matplotlib.pyplot as plt
    BLUE, GREY, LIGHT, INK, MUTED = "#2a78d6", "#a3a19b", "#e4e2dd", "#0b0b0b", "#52514e"
    panels = [("× cheaper per claim", lambda r: BASELINE["cost_per_claim"] / r["cost_per_claim"]),
              ("× faster per claim (p50)", lambda r: BASELINE["p50"] / r["p50"]),
              ("× faster for all claims", lambda r: BASELINE["wall"] / r["wall"])]
    runs = runs[::-1]                                              # first run on top
    fig, axes = plt.subplots(1, 3, figsize=(13, 0.5 * len(runs) + 1.8), sharey=True, facecolor="#fcfcfb")
    for ax, (name, value) in zip(axes, panels):
        vals = [value(r) for _, r in runs]
        failed = [r["accuracy"] < 0.9 for _, r in runs]
        colours = [GREY if label == "v0" else LIGHT if bad else BLUE for (label, _), bad in zip(runs, failed)]
        # Levers multiply (×3, then ×2, ...), so the axis is logarithmic: every lever gets a visible step.
        bars = ax.barh(range(len(runs)), [v - 1 for v in vals], left=1, color=colours, height=0.62)
        for bar, bad in zip(bars, failed):
            if bad:
                bar.set_hatch("///")
                bar.set_edgecolor(GREY)
        ax.bar_label(bars, [f"×{v:,.0f}" if v >= 10 else f"×{v:.1f}" for v in vals], padding=4, color=INK, fontsize=9)
        ax.set_title(name, loc="left", fontsize=10, color=INK)
        ax.set_facecolor("#fcfcfb")
        ax.grid(axis="x", color=LIGHT, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.tick_params(colors=MUTED, labelsize=9, length=0)
        ax.tick_params(which="minor", length=0)
        ax.set_xscale("log")
        upper = max(vals) * 3
        ax.set_xlim(0.8, upper)
        steps = (1, 2, 5, 10, 20, 50) if upper <= 60 else (1, 10, 100, 1000, 10000)
        ax.set_xticks([x for x in steps if x <= upper])
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"×{x:g}"))
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.axvline(1, color=GREY, linewidth=1)
        for side in ("top", "right", "bottom"):
            ax.spines[side].set_visible(False)
        ax.spines["left"].set_color(LIGHT)
    axes[0].set_yticks(range(len(runs)))
    axes[0].set_yticklabels([label + ("  (below 90 %)" if r["accuracy"] < 0.9 else "") for label, r in runs])
    fig.suptitle(title, x=0.01, ha="left", fontsize=12, color=INK)
    fig.text(0.01, 0.01, "v0 = ×1 (grey line). Logarithmic axis. Hatched runs got less than 90 % of the claims right, so they don't count.",
             fontsize=8.5, color=MUTED)
    fig.tight_layout(rect=(0, 0.04, 1, 0.97))
    plt.show()


def scorecard(claims=CLAIMS, label="inference_v1"):
    if CONFIG == V0_CONFIG and claims is CLAIMS:
        print("These are the v0 settings – showing the recorded v0 run instead of paying for it again.\n")
        rep = BASELINE
    else:
        rep = run_claims(inference_v1, claims, workers=CONFIG["workers"])
        if claims is CLAIMS:
            RUN_HISTORY.append((config_label(), rep))
    report(rep, label)
    cheaper = BASELINE["cost_per_claim"] / rep["cost_per_claim"]
    faster = BASELINE["p50"] / rep["p50"]
    if rep["accuracy"] < 0.9:
        print(f"\n⛔ accuracy {rep['accuracy']:.0%} – below 90 %, so this run doesn't count.")
    else:
        print(f"\n✓ accuracy {rep['accuracy']:.0%} · {cheaper:,.0f}× cheaper · {faster:,.1f}× faster than v0")
    return rep


print("Scorecard ready.")

In [ ]:
# ▶ RUN – your result (with v0 settings it shows the recorded run for free; an optimised run costs a few cents)
my_run = scorecard()

In [ ]:
# ▶ RUN – all your runs so far, compared with v0 (free – no API calls)
if RUN_HISTORY:
    plot_runs([("v0", BASELINE)] + RUN_HISTORY, "Your runs compared with v0")
else:
    print("No runs yet – change CONFIG and run the scorecard above first.")

Change `CONFIG`, run it and the scorecard again, and look at the chart. Repeat. A good order to try:
`one_call` → `structured` → `cache` → `routing` → `workers` → `effort`. Then experiment: is Haiku good enough for everything?
Can you get close to Haiku's price without losing accuracy?

**The holdout.** When you're done, test your pipeline on three claims it has never seen. A pipeline
that only works on the fourteen sample claims has learned the samples, not the job.

In [ ]:
# ✏️ YOUR TURN – set to True when your CONFIG is final
RUN_HOLDOUT = False

HOLDOUT = [
    {"id": "H-01", "channel": "form", "customer": "Hanna Wirth", "received": "2027-01-07",
     "order": "ORD-09720: Decaf 1 kg €34.90 · order total €34.90, shipping €4.90 charged",
     "claimed": "Decaf 1 kg", "delivered": "2026-11-26", "opened": "no", "dropdown": "changed my mind", "system_refund": 39.80,
     "description": "",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "outside_return_window"}},
    {"id": "H-02", "channel": "email", "customer": "Lisa Hofmann", "received": "2027-01-06",
     "email": "Hello, order IC-1990. I put my moka pot on the stove and forgot the water. The rubber ring has "
              "melted and the bottom is discoloured. Is this covered? Lisa",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "misuse"}},
    {"id": "H-03", "channel": "email", "customer": "Daniel Fuchs", "received": "2027-01-05",
     "email": "Hello, my Hand Grinder (order IC-2101) is stuck and the crank won't turn any more. It's only three "
              "months old! I mostly use it for coffee, now and then for pepper when my pepper mill is empty. "
              "Please send me a new one. Daniel",
     "gold": {"decision": "reject", "refund_eur": 0, "reason": "misuse"}},
]
if RUN_HOLDOUT:
    scorecard(HOLDOUT, "holdout")
else:
    print("Holdout ready. Set RUN_HOLDOUT = True when your CONFIG is final – and don't tune against it.")

## ⭐ Extensions

- ⭐ **Lever by lever.** The first cell below switches the levers on one after another, starting
  from v0 (each step keeps the ones before), and draws the chart. Which lever saved the most money?
  Which saved the most time?
- ⭐ **A year of claims.** Isar Coffee expects about 12,000 claims a year, 20 % of them while a
  customer is on the phone (interactive) and 80 % from the overnight queue (batch, 50 % off).
  The second cell compares a year of v0 with a year of your pipeline – in waiting time and in money.
- ⭐⭐ **Cache lifetime.** On Sundays a claim arrives only every 20 minutes, so the 5-minute cache
  expires in between. The API also offers a 1-hour cache (`"ttl": "1h"`), which costs twice the
  input price to write instead of 1.25×. Which is cheaper for Sunday traffic?
- ⭐⭐ **Batch for real.** With your own Anthropic key, set `RUN_BATCH = True` in Lever 7 and fetch
  the results later with `client.messages.batches.results(batch_id)`.

In [ ]:
# ✏️ YOUR TURN – lever by lever (⭐). Set to True to run it: about 6 minutes, about $0.80
# (the first two steps still run on Opus).
RUN_LADDER = False

STEPS = [("+ 1 call", {"one_call": True}), ("+ JSON record", {"structured": True}),
         ("+ cache", {"cache": True}), ("+ routing", {"routing": True}), ("+ 8 parallel", {"workers": 8})]
if RUN_LADDER:
    _saved = dict(CONFIG)
    ladder = [("v0", BASELINE)]
    try:
        CONFIG.clear()
        CONFIG.update(V0_CONFIG)
        for label, change in STEPS:
            CONFIG.update(change)
            print(f"running {label} ...", flush=True)
            ladder.append((label, run_claims(inference_v1, workers=CONFIG["workers"])))
    finally:
        CONFIG.clear()
        CONFIG.update(_saved)                                   # your own CONFIG stays as it was
    plot_runs(ladder, "Lever by lever – each step keeps the levers before it")
else:
    print("Set RUN_LADDER = True to run the levers one by one and see the chart.")

In [ ]:
# ✏️ YOUR TURN – a year of claims (⭐)
CLAIMS_PER_YEAR = 12_000
INTERACTIVE_SHARE = 0.20      # claims handled while the customer waits; the rest goes through the batch lane

phone_claims = CLAIMS_PER_YEAR * INTERACTIVE_SHARE
v0_money = CLAIMS_PER_YEAR * BASELINE["cost_per_claim"]                       # v0 has no batch lane
your_money = CLAIMS_PER_YEAR * my_run["cost_per_claim"] * (INTERACTIVE_SHARE + (1 - INTERACTIVE_SHARE) * 0.5)
print(f"v0   customers wait {phone_claims * BASELINE['p50'] / 3600:6.1f} hours in total · ${v0_money:,.2f} per year")
print(f"you  customers wait {phone_claims * my_run['p50'] / 3600:6.1f} hours in total · ${your_money:,.2f} per year (with the batch lane)")